In [5]:
import os
import json
from glob import glob
import pandas as pd

# PATH
BASE_DIR = "/kaggle/input/datasets/wildanhasanahfitrah/oil-and-gas-2/Data/Test_with_anomalies"
OUT_DIR  = "/kaggle/working/excel_anomaly_output"

os.makedirs(OUT_DIR, exist_ok=True)

# FUNCTION PARSING
def parse_anomaly_json(json_path):

    with open(json_path, "r") as f:
        js = json.load(f)

    sim     = js.get("Simulation", {})
    sensors = js.get("Sensors", {})
    valves  = js.get("Pneumatic_Valves", {})
    pid     = js.get("PID_Controllers", {})

    # TIME
    time = sim.get("sim_time", [])
    n = len(time)

    df = pd.DataFrame({"Time": time})

    # SENSORS S1–S8
    for s in [f"S{i}" for i in range(1,9)]:
        if s in sensors and len(sensors[s]) == n:
            df[s] = sensors[s]

    # VALVES
    valve_keys = ["AV1","AV2","AV3","AV2_PP","AV2_LD","AV3_PP","AV3_LD"]

    for k in valve_keys:
        v = valves.get(k)
        if isinstance(v, list) and len(v) == n:
            df[k] = v

    # PID CONTROLLERS 
    for i in [1,2,3]:
        pid_key = f"PID{i}"

        if pid_key in pid:
            header = pid[pid_key].get("Header", [])
            values = pid[pid_key].get("Values", [])

            if values and len(values) == n:
                pid_df = pd.DataFrame(values, columns=header)

                for col in header:
                    df[col] = pid_df[col].values

    # ANOMALY FLAGS VM1–VM7
    for vm in [f"VM{i}" for i in range(1,8)]:
        df[vm] = 0

    anomaly_name = sim.get("anomaly")
    anomaly_time = sim.get("anomaly_time")

    if isinstance(anomaly_name, str) and anomaly_name.startswith("VM"):

        arr = (anomaly_time[:n] + [0]*n)[:n]
        df[anomaly_name] = arr

        # label per timestep
        df["label"] = df[anomaly_name]
    else:
        df["label"] = 0

    # TAMBAHAN INFO
    folder_name = os.path.basename(os.path.dirname(json_path))
    df["anomaly_type"] = folder_name

    # FIX FORMAT ANGKA
    df = df.replace(",", ".", regex=True)

    for col in df.columns:
        if col not in ["Time", "anomaly_type"]:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    # SAVE
    relative_path = os.path.relpath(json_path, BASE_DIR)
    out_path = os.path.join(OUT_DIR, relative_path.replace(".json", ".xlsx"))

    os.makedirs(os.path.dirname(out_path), exist_ok=True)

    df.to_excel(out_path, index=False, sheet_name="Sheet1")

    return out_path

# PROCESS SEMUA FILE
outputs = []

json_files = glob(os.path.join(BASE_DIR, "**", "*.json"), recursive=True)

for json_file in sorted(json_files):
    out = parse_anomaly_json(json_file)
    outputs.append(out)

print("Total files processed:", len(outputs))

Total files processed: 21


In [6]:
!zip -r /kaggle/working/excel_anomaly_output.zip /kaggle/working/excel_anomaly_output

updating: kaggle/working/excel_anomaly_output/ (stored 0%)
updating: kaggle/working/excel_anomaly_output/anomalyVM1/ (stored 0%)
updating: kaggle/working/excel_anomaly_output/anomalyVM1/anomalyVM1_severe/ (stored 0%)
updating: kaggle/working/excel_anomaly_output/anomalyVM1/anomalyVM1_severe/anomalyVM1_severe.xlsx (deflated 15%)
updating: kaggle/working/excel_anomaly_output/anomalyVM1/anomalyVM1_medium/ (stored 0%)
updating: kaggle/working/excel_anomaly_output/anomalyVM1/anomalyVM1_medium/anomalyVM1_medium.xlsx (deflated 6%)
updating: kaggle/working/excel_anomaly_output/anomalyVM1/anomalyVM1_slight/ (stored 0%)
updating: kaggle/working/excel_anomaly_output/anomalyVM1/anomalyVM1_slight/anomalyVM1_slight.xlsx (deflated 4%)
updating: kaggle/working/excel_anomaly_output/anomalyVM5/ (stored 0%)
updating: kaggle/working/excel_anomaly_output/anomalyVM5/anomalyVM5_slight/ (stored 0%)
updating: kaggle/working/excel_anomaly_output/anomalyVM5/anomalyVM5_slight/anomalyVM5_slight.xlsx (deflated 5%)


In [4]:
import shutil
import os

OUT_DIR = "/kaggle/working/excel_anomaly_output"

if os.path.exists(OUT_DIR):
    shutil.rmtree(OUT_DIR)
    print("Output folder deleted.")
else:
    print("Output folder does not exist.")


Output folder deleted.
